In [0]:

-- 1. Staging view to normalize dates
CREATE STREAMING LIVE VIEW stg_promotions AS
SELECT
  promotion_id,
  promotion_name,
  promotion_type,
  discount_percent,
  -- Handle malformed start_date strings
  COALESCE(
    try_to_date(start_date, 'MM/dd/yyyy'),
    try_to_date(start_date, 'yyyy-MM-dd'),
    try_to_date(start_date, 'dd-MMM-yy')
  ) AS start_date,
  end_date, -- Already inferred as DATE type in Bronze
  channel,
  media_type,
  ingested_timestamp,
  batch_date
FROM STREAM(mavenmarket.bronze.promotions);

-- 2. Clean Silver Table filtering out all intentional anomalies
CREATE STREAMING LIVE TABLE silver_promotions (
  CONSTRAINT valid_id EXPECT (promotion_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_discount EXPECT (discount_percent >= 0 AND discount_percent <= 1.0) ON VIOLATION DROP ROW,
  CONSTRAINT valid_dates EXPECT (end_date >= start_date) ON VIOLATION DROP ROW
)
COMMENT "Cleaned promotions data"
AS SELECT * FROM STREAM(live.stg_promotions);